## Fairness index
There a 3 thread groups representing 3 user groups requesting a set of resources concurrently. We need to compare theoretical throughput decided from jmeter with the effective throughput measured.

In [1]:
import pandas as pd

fname = "fairness_index_out.csv"

# 800 capacity
thread_groups = {
    "TG1": 200/60,
    "TG2": 200/60,
    "TG3": 400/60
}

df = pd.read_csv(fname)

df.head()


,timeStamp,elapsed,label,responseCode,responseMessage,threadName,dataType,success,failureMessage,bytes,sentBytes,grpThreads,allThreads,URL,Latency,IdleTime,Connect
0,1704830072719,169,2,200,OK,TG2 2-2,text,True,NaN,130275,115,3,11,http://192.168.1.53/2.html,84,0,70
1,1704830072685,213,2,200,OK,TG1 1-2,text,True,NaN,130275,115,7,11,http://192.168.1.53/2.html,118,0,104
2,1704830072685,278,3,200,OK,TG1 1-1,text,True,NaN,236294,115,8,12,http://192.168.1.53/3.docx,129,0,104
3,1704830072685,289,3,200,OK,TG2 2-1,text,True,NaN,236294,115,3,13,http://192.168.1.53/3.docx,130,0,104
4,1704830072717,262,5,200,OK,TG1 1-4,text,True,NaN,463401,114,9,13,http://192.168.1.53/5.pdf,86,0,72


# Measuring throughput
We need to measure the throughput for each thread groupd. So, we apply the troughput formula for each thread groups $i$.

\begin{equation}
    t = \frac{requests_i}{duration}
\end{equation}

In [2]:
# get duration in seconds
duration = (df["timeStamp"].max() - df["timeStamp"].min()) / 1000

# extract thread group
df["threadGroup"] = df["threadName"].apply(lambda x: str(x).split(" ")[0])

for k, ft in thread_groups.items():
    mt = df[df.threadGroup == k].shape[0] / duration
    print(mt)
    thread_groups[k] = mt / ft

3.3876603803782444
3.377657446184214
6.535250340099763


# Fair index
Compute the fair index based on the normaliztion index computed before.

\begin{equation}
    f = \frac{\left(\sum \limits _{i=1}^3 x_i\right)^2}{3 \cdot \sum \limits _{i=1}^3 x_i^2}
\end{equation}

In [39]:
import numpy as np
norm = np.array(list(thread_groups.values()))

(norm.sum() ** 2) / (norm.size * np.power(norm, 2).sum())

0.9997356587413978